## Tập NASA

## Nasa (05 tập)

In [7]:
# ============================================================
# Software Fault Prediction - NASA PROMISE (FINAL CLEAN)
# 6 Metaheuristics + 6 Models + Full Metrics + No Warning
# ============================================================

import numpy as np
import pandas as pd
import os
import warnings

# ===== TẮT WARNING =====
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import HistGradientBoostingClassifier

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# ==========================
# ELM
# ==========================
class ELM:
    def __init__(self, input_size, hidden_size=50):
        self.input_size = input_size
        self.hidden_size = hidden_size

    def _act(self, x):
        return 1/(1+np.exp(-x))

    def fit(self, X, y):
        self.W = np.random.randn(self.input_size, self.hidden_size)
        self.b = np.random.randn(self.hidden_size)
        H = self._act(X @ self.W + self.b)
        self.beta = np.linalg.pinv(H) @ y

    def predict(self, X):
        H = self._act(X @ self.W + self.b)
        y = H @ self.beta
        return (y > 0.5).astype(int)

    def predict_proba(self, X):
        H = self._act(X @ self.W + self.b)
        return H @ self.beta


# ============================================================
# 🔥 LOAD DATA (FIX ALL NASA ISSUES)
# ============================================================
def load_data(path):
    df = pd.read_csv(path).dropna()

    y = df.iloc[:, -1]

    # label xử lý
    if y.dtype == 'object':
        y = y.astype(str).str.lower().str.strip()
        if set(y.unique()).issubset({'yes', 'no'}):
            y = y.map({'yes': 1, 'no': 0})
        else:
            y = pd.factorize(y)[0]
    else:
        y = (y > 0).astype(int)

    X = df.iloc[:, :-1]
    X = X.select_dtypes(include=[np.number]).values

    return X, y.values


# ==========================
# Fitness
# ==========================

def fitness(X, y, sol):
    # chọn top-6 features
    idx = np.argsort(sol)[-6:]
    X_sel = X[:, idx]

    # ===== scoring (không dùng ML) =====
    scores = np.mean(X_sel, axis=1)

    # normalize về [0,1]
    scores = (scores - scores.min()) / (scores.max() - scores.min() + 1e-9)

    # threshold
    preds = (scores >= 0.5).astype(int)

    # ===== confusion matrix =====
    TP = np.sum((y == 1) & (preds == 1))
    TN = np.sum((y == 0) & (preds == 0))
    FP = np.sum((y == 0) & (preds == 1))
    FN = np.sum((y == 1) & (preds == 0))

    # ===== tránh chia 0 =====
    recall = TP / (TP + FN + 1e-9)
    specificity = TN / (TN + FP + 1e-9)

    return recall + specificity

def fitness1(X, y, sol):
    idx = np.argsort(sol)[-6:]
    X_sel = X[:, idx]

    model = MLPClassifier(hidden_layer_sizes=(30,), max_iter=200)
    model.fit(X_sel, y)
    pred = model.predict(X_sel)

    return f1_score(y, pred, average='weighted')


# ==========================
# Metaheuristics
# ==========================
def init_pop(n, d): return np.random.rand(n, d)

def GA(f, d, it=20, n=10):
    P = init_pop(n, d)
    for _ in range(it):
        P = np.array(sorted(P, key=f)[-n//2:])
        C = []
        for _ in range(n//2):
            i, j = np.random.randint(len(P)), np.random.randint(len(P))
            cp = np.random.randint(d)
            C.append(np.concatenate([P[i][:cp], P[j][cp:]]))
        P = np.vstack((P, C))
        P += np.random.normal(0, 0.1, P.shape)
        P = np.clip(P, 0, 1)
    return max(P, key=f)

def PSO(f, d, it=20, n=10):
    X = init_pop(n, d); V = np.zeros_like(X); p = X.copy()
    for _ in range(it):
        for i in range(n):
            if f(X[i]) > f(p[i]): p[i] = X[i]
        g = max(p, key=f)
        for i in range(n):
            r1, r2 = np.random.rand(d), np.random.rand(d)
            V[i] = 0.5*V[i] + 1.5*r1*(p[i]-X[i]) + 1.5*r2*(g-X[i])
            X[i] = np.clip(X[i] + V[i], 0, 1)
    return g

def GWO(f, d, it=20, n=10):
    X = init_pop(n, d)
    for t in range(it):
        X = sorted(X, key=f)
        a = 2 - 2*t/it
        for i in range(n):
            for leader in X[-3:]:
                A = 2*a*np.random.rand(d)-a
                C = 2*np.random.rand(d)
                D = abs(C*leader - X[i])
                X[i] = leader - A*D
        X = np.clip(X, 0, 1)
    return max(X, key=f)

def Squirrel(f, d, it=20, n=10):
    X = init_pop(n, d)
    for _ in range(it):
        best = max(X, key=f)
        for i in range(n):
            X[i] += np.random.rand(d)*(best - X[i])
        X = np.clip(X, 0, 1)
    return best

def MRA(f, d, it=20, n=10):
    X = init_pop(n, d)
    for _ in range(it):
        fit = np.array([f(x) for x in X])
        probs = fit / (fit.sum()+1e-9)
        newX = []
        for _ in range(n):
            i = np.random.choice(range(n), p=probs)
            newX.append(X[i] + np.random.normal(0, 0.1, d))
        X = np.clip(np.array(newX), 0, 1)
    return max(X, key=f)

def SSA(f, d, it=20, n=10):
    X = init_pop(n, d)
    for t in range(it):
        c1 = 2*np.exp(-(4*t/it)**2)
        food = max(X, key=f)
        for i in range(n):
            if i == 0:
                X[i] = food + c1*(np.random.rand(d)-0.5)
            else:
                X[i] = (X[i] + X[i-1]) / 2
        X = np.clip(X, 0, 1)
    return max(X, key=f)

def init_pop1(n, d, lb, ub):
    return lb + (ub - lb) * np.random.rand(n, d)

def BBOA(f, d, it=100, n=30, lb=0, ub=1):
    X = init_pop1(n, d, lb, ub)
    best = max(X, key=f)
    
    for t in range(it):
        a = 2 * (1 - t/it)
        p = 1 - t/it
        
        for i in range(n):
            if np.random.rand() < p:
                # Exploration
                X_rand = X[np.random.randint(n)]
                r1 = np.random.rand(d)
                r2 = np.random.rand(d)
                X[i] = X[i] + r1 * (X_rand - r2 * X[i])
            else:
                # Exploitation
                r = np.random.rand(d)
                X[i] = best + a * r * (X[i] - best)
            
            # Boundary control
            X[i] = np.clip(X[i], lb, ub)
        
        # Update best
        current_best = max(X, key=f)
        if f(current_best) > f(best):
            best = current_best
    
    return best
algos = {
    
}
algos1 = {
    "GA": GA, "Squirrel": Squirrel, "GWO": GWO,
    "MRA": MRA, "PSO": PSO, "SSA": SSA, "BBOA": BBOA
}


# ==========================
# Models
# ==========================
def get_models(input_dim):
    return {
        "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss'),
        "LightGBM": LGBMClassifier(n_estimators=200, 
                                   learning_rate=0.05,verbose=-1,random_state=42),
        "CatBoost": CatBoostClassifier(iterations=200, verbose=0),
        "HGBoost": HistGradientBoostingClassifier(max_iter=200),
        "MLP": MLPClassifier(hidden_layer_sizes=(50,), max_iter=500, early_stopping=True),
        "ELM": ELM(input_dim)
    }


# ==========================
# Evaluation
# ==========================
def evaluate(model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)

    try:
        y_prob = model.predict_proba(X_test)
        if len(y_prob.shape) > 1:
            y_prob = y_prob[:,1]
    except:
        try:
            y_prob = model.decision_function(X_test)
        except:
            y_prob = model.predict(X_test)

    y_pred = model.predict(X_test)

    try:
        auc = roc_auc_score(y_test, y_prob)
    except:
        auc = 0.5

    return {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, average='weighted'),
        "Recall": recall_score(y_test, y_pred, average='weighted'),
        "F1": f1_score(y_test, y_pred, average='weighted'),
        "AUC": auc
    }

def run_all_baseline(folder=r"E:\DEU\DUE Scopus\NASA-promise-dataset-repository-main"):
    results = []

    for file in sorted(os.listdir(folder)):
        if not file.endswith(".csv"):
            continue

        print("\nProcessing:", file)

        try:
            # Load data
            X, y = load_data(os.path.join(folder, file))

            # Scale
            scaler = MinMaxScaler()
            X = scaler.fit_transform(X)

            # Handle stratify
            unique, counts = np.unique(y, return_counts=True)

            if np.min(counts) < 2:
                X_train, X_test, y_train, y_test = train_test_split(
                    X, y, test_size=0.3, random_state=42
                )
            else:
                X_train, X_test, y_train, y_test = train_test_split(
                    X, y, test_size=0.3, random_state=42, stratify=y
                )

            # ❌ BỎ hoàn toàn phần algo + chọn feature

            # ✅ Lấy tất cả feature
            models = get_models(X.shape[1])

            for model_name, model in models.items():
                metrics = evaluate(model, X_train, X_test, y_train, y_test)

                res = {
                    "Dataset": file,
                    "Model": model_name,
                    **{k: round(v, 4) for k, v in metrics.items()}
                }

                results.append(res)
                print(res)

        except Exception as e:
            print(f"Error processing {file}: {e}")

    return results
# ==========================
# MAIN
# ==========================
def run_all(folder=r"E:\DEU\DUE Scopus\NASA-promise-dataset-repository-main"):
    results = []

    for file in sorted(os.listdir(folder)):
        if not file.endswith(".csv"): continue
        print("\nProcessing:", file)

        try:
            X, y = load_data(os.path.join(folder, file))

            scaler = MinMaxScaler()
            X = scaler.fit_transform(X)

            unique, counts = np.unique(y, return_counts=True)

            if np.min(counts) < 2:
                X_train, X_test, y_train, y_test = train_test_split(
                    X, y, test_size=0.3, random_state=42
                )
            else:
                X_train, X_test, y_train, y_test = train_test_split(
                    X, y, test_size=0.3, random_state=42, stratify=y
                )

            d = X.shape[1]

            for algo_name, algo in algos.items():

                sol = algo(lambda s: fitness(X_train, y_train, s), d)
                idx = np.argsort(sol)[-6:]

                Xtr = X_train[:, idx]
                Xte = X_test[:, idx]

                models = get_models(len(idx))

                for model_name, model in models.items():
                    metrics = evaluate(model, Xtr, Xte, y_train, y_test)

                    res = {
                        "Dataset": file,
                        "Algo": algo_name,
                        "Model": model_name,
                        **{k: round(v,4) for k,v in metrics.items()}
                    }

                    results.append(res)
                    print(res)

        except Exception as e:
            print("Error:", file, e)

    
    
    
    df = pd.DataFrame(results)
    df.to_excel("SFP_results_nasa_baseline.xlsx", index=False)

    print("\n=== SUMMARY ===")
    #print(df.groupby(["Algo","Model"])["F1"].mean())
    summary = df.groupby(["Algo","Model"])[['Accuracy', 'Precision', 'Recall', 'F1', 'AUC']].mean()
    print(summary)
    summary.to_excel('SFP_summary_nasa.xlsx')


# ==========================
# RUN
# ==========================
run_all_baseline()


Processing: cm1.csv
{'Dataset': 'cm1.csv', 'Model': 'XGBoost', 'Accuracy': 0.8667, 'Precision': 0.8496, 'Recall': 0.8667, 'F1': 0.8574, 'AUC': 0.6025}
{'Dataset': 'cm1.csv', 'Model': 'LightGBM', 'Accuracy': 0.8733, 'Precision': 0.8616, 'Recall': 0.8733, 'F1': 0.867, 'AUC': 0.6188}
{'Dataset': 'cm1.csv', 'Model': 'CatBoost', 'Accuracy': 0.8867, 'Precision': 0.8088, 'Recall': 0.8867, 'F1': 0.8459, 'AUC': 0.6321}
{'Dataset': 'cm1.csv', 'Model': 'HGBoost', 'Accuracy': 0.8467, 'Precision': 0.8319, 'Recall': 0.8467, 'F1': 0.839, 'AUC': 0.6356}
{'Dataset': 'cm1.csv', 'Model': 'MLP', 'Accuracy': 0.9, 'Precision': 0.81, 'Recall': 0.9, 'F1': 0.8526, 'AUC': 0.4627}
{'Dataset': 'cm1.csv', 'Model': 'ELM', 'Accuracy': 0.8533, 'Precision': 0.8056, 'Recall': 0.8533, 'F1': 0.8288, 'AUC': 0.682}

Processing: jm1.csv
{'Dataset': 'jm1.csv', 'Model': 'XGBoost', 'Accuracy': 0.8483, 'Precision': 0.8184, 'Recall': 0.8483, 'F1': 0.8187, 'AUC': 0.745}
{'Dataset': 'jm1.csv', 'Model': 'LightGBM', 'Accuracy': 0.8

[{'Dataset': 'cm1.csv',
  'Model': 'XGBoost',
  'Accuracy': 0.8667,
  'Precision': 0.8496,
  'Recall': 0.8667,
  'F1': 0.8574,
  'AUC': 0.6025},
 {'Dataset': 'cm1.csv',
  'Model': 'LightGBM',
  'Accuracy': 0.8733,
  'Precision': 0.8616,
  'Recall': 0.8733,
  'F1': 0.867,
  'AUC': 0.6188},
 {'Dataset': 'cm1.csv',
  'Model': 'CatBoost',
  'Accuracy': 0.8867,
  'Precision': 0.8088,
  'Recall': 0.8867,
  'F1': 0.8459,
  'AUC': 0.6321},
 {'Dataset': 'cm1.csv',
  'Model': 'HGBoost',
  'Accuracy': 0.8467,
  'Precision': 0.8319,
  'Recall': 0.8467,
  'F1': 0.839,
  'AUC': 0.6356},
 {'Dataset': 'cm1.csv',
  'Model': 'MLP',
  'Accuracy': 0.9,
  'Precision': 0.81,
  'Recall': 0.9,
  'F1': 0.8526,
  'AUC': 0.4627},
 {'Dataset': 'cm1.csv',
  'Model': 'ELM',
  'Accuracy': 0.8533,
  'Precision': 0.8056,
  'Recall': 0.8533,
  'F1': 0.8288,
  'AUC': 0.682},
 {'Dataset': 'jm1.csv',
  'Model': 'XGBoost',
  'Accuracy': 0.8483,
  'Precision': 0.8184,
  'Recall': 0.8483,
  'F1': 0.8187,
  'AUC': 0.745},
 {'

## Tập Apache + SMOTE

In [6]:
# ============================================================
# Software Fault Prediction - NASA PROMISE (FINAL CLEAN)
# 6 Metaheuristics + 6 Models + Full Metrics + No Warning
# ============================================================

import numpy as np
import pandas as pd
import os
import warnings

# ===== TẮT WARNING =====
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import HistGradientBoostingClassifier

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from imblearn.over_sampling import SMOTE
# ==========================
# ELM
# ==========================
class ELM:
    def __init__(self, input_size, hidden_size=50):
        self.input_size = input_size
        self.hidden_size = hidden_size

    def _act(self, x):
        return 1/(1+np.exp(-x))

    def fit(self, X, y):
        self.W = np.random.randn(self.input_size, self.hidden_size)
        self.b = np.random.randn(self.hidden_size)
        H = self._act(X @ self.W + self.b)
        self.beta = np.linalg.pinv(H) @ y

    def predict(self, X):
        H = self._act(X @ self.W + self.b)
        y = H @ self.beta
        return (y > 0.5).astype(int)

    def predict_proba(self, X):
        H = self._act(X @ self.W + self.b)
        return H @ self.beta


# ==========================
# Load data (FIX binary)
# ==========================
def load_data(path):
    df = pd.read_csv(path).dropna()
    df = df.select_dtypes(include=[np.number])

    X = df.iloc[:, :-1].values
    y = df.iloc[:, -1].values

    # Convert defect true/false → binary
    y = y.astype(int)

    return X, y


# ==========================
# Fitness
# ==========================

def fitness(X, y, sol):
    # chọn top-6 features
    idx = np.argsort(sol)[-6:]
    X_sel = X[:, idx]

    # ===== scoring (không dùng ML) =====
    scores = np.mean(X_sel, axis=1)

    # normalize về [0,1]
    scores = (scores - scores.min()) / (scores.max() - scores.min() + 1e-9)

    # threshold
    preds = (scores >= 0.5).astype(int)

    # ===== confusion matrix =====
    TP = np.sum((y == 1) & (preds == 1))
    TN = np.sum((y == 0) & (preds == 0))
    FP = np.sum((y == 0) & (preds == 1))
    FN = np.sum((y == 1) & (preds == 0))

    # ===== tránh chia 0 =====
    recall = TP / (TP + FN + 1e-9)
    specificity = TN / (TN + FP + 1e-9)

    return recall + specificity

def fitness1(X, y, sol):
    idx = np.argsort(sol)[-6:]
    X_sel = X[:, idx]

    model = MLPClassifier(hidden_layer_sizes=(30,), max_iter=200)
    model.fit(X_sel, y)
    pred = model.predict(X_sel)

    return f1_score(y, pred, average='weighted')


# ==========================
# Metaheuristics
# ==========================
def init_pop(n, d): return np.random.rand(n, d)

def GA(f, d, it=20, n=10):
    P = init_pop(n, d)
    for _ in range(it):
        P = np.array(sorted(P, key=f)[-n//2:])
        C = []
        for _ in range(n//2):
            i, j = np.random.randint(len(P)), np.random.randint(len(P))
            cp = np.random.randint(d)
            C.append(np.concatenate([P[i][:cp], P[j][cp:]]))
        P = np.vstack((P, C))
        P += np.random.normal(0, 0.1, P.shape)
        P = np.clip(P, 0, 1)
    return max(P, key=f)

def PSO(f, d, it=20, n=10):
    X = init_pop(n, d); V = np.zeros_like(X); p = X.copy()
    for _ in range(it):
        for i in range(n):
            if f(X[i]) > f(p[i]): p[i] = X[i]
        g = max(p, key=f)
        for i in range(n):
            r1, r2 = np.random.rand(d), np.random.rand(d)
            V[i] = 0.5*V[i] + 1.5*r1*(p[i]-X[i]) + 1.5*r2*(g-X[i])
            X[i] = np.clip(X[i] + V[i], 0, 1)
    return g

def GWO(f, d, it=20, n=10):
    X = init_pop(n, d)
    for t in range(it):
        X = sorted(X, key=f)
        a = 2 - 2*t/it
        for i in range(n):
            for leader in X[-3:]:
                A = 2*a*np.random.rand(d)-a
                C = 2*np.random.rand(d)
                D = abs(C*leader - X[i])
                X[i] = leader - A*D
        X = np.clip(X, 0, 1)
    return max(X, key=f)

def Squirrel(f, d, it=20, n=10):
    X = init_pop(n, d)
    for _ in range(it):
        best = max(X, key=f)
        for i in range(n):
            X[i] += np.random.rand(d)*(best - X[i])
        X = np.clip(X, 0, 1)
    return best

def MRA(f, d, it=20, n=10):
    X = init_pop(n, d)
    for _ in range(it):
        fit = np.array([f(x) for x in X])
        probs = fit / (fit.sum()+1e-9)
        newX = []
        for _ in range(n):
            i = np.random.choice(range(n), p=probs)
            newX.append(X[i] + np.random.normal(0, 0.1, d))
        X = np.clip(np.array(newX), 0, 1)
    return max(X, key=f)

def SSA(f, d, it=20, n=10):
    X = init_pop(n, d)
    for t in range(it):
        c1 = 2*np.exp(-(4*t/it)**2)
        food = max(X, key=f)
        for i in range(n):
            if i == 0:
                X[i] = food + c1*(np.random.rand(d)-0.5)
            else:
                X[i] = (X[i] + X[i-1]) / 2
        X = np.clip(X, 0, 1)
    return max(X, key=f)

algos = {
    "GA": GA, "Squirrel": Squirrel, "GWO": GWO,
    "MRA": MRA, "PSO": PSO, "SSA": SSA
}


# ==========================
# Models
# ==========================
def get_models(input_dim):
    return {
        "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss'),
        "LightGBM": LGBMClassifier(n_estimators=200, 
                                   learning_rate=0.05,verbose=-1,random_state=42),
        "CatBoost": CatBoostClassifier(iterations=200, verbose=0),
        "HGBoost": HistGradientBoostingClassifier(max_iter=200),
        "MLP": MLPClassifier(hidden_layer_sizes=(50,), max_iter=500, early_stopping=True),
        "ELM": ELM(input_dim)
    }


# ==========================
# SMOTE
# ==========================
def apply_smote(X, y):
    unique, counts = np.unique(y, return_counts=True)

    # nếu dữ liệu quá ít → bỏ qua
    if len(counts) < 2 or np.min(counts) < 2:
        print("SMOTE skipped: too few samples")
        return X, y

    # tự động điều chỉnh k_neighbors
    k = min(5, np.min(counts) - 1)

    try:
        smote = SMOTE(k_neighbors=k, random_state=42)
        X_res, y_res = smote.fit_resample(X, y)
        print(f"SMOTE applied (k={k})")
        return X_res, y_res
    except Exception as e:
        print("SMOTE failed:", e)
        return X, y


# ==========================
# Evaluation
# ==========================
def evaluate(model, X_train, X_test, y_train, y_test):
    
    # ===== SMOTE chỉ áp dụng trên TRAIN =====
    X_train, y_train = apply_smote(X_train, y_train)
    model.fit(X_train, y_train)

    try:
        y_prob = model.predict_proba(X_test)
        if len(y_prob.shape) > 1:
            y_prob = y_prob[:,1]
    except:
        try:
            y_prob = model.decision_function(X_test)
        except:
            y_prob = model.predict(X_test)

    y_pred = model.predict(X_test)

    try:
        auc = roc_auc_score(y_test, y_prob)
    except:
        auc = 0.5

    return {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, average='weighted'),
        "Recall": recall_score(y_test, y_pred, average='weighted'),
        "F1": f1_score(y_test, y_pred, average='weighted'),
        "AUC": auc
    }


# ==========================
# MAIN
# ==========================
def run_all(folder="/kaggle/input/datasets/hovanphi/promise"): #Apache
    results = []

    for file in sorted(os.listdir(folder)):
        if not file.endswith(".csv"): continue
        print("\nProcessing:", file)

        try:
            X, y = load_data(os.path.join(folder, file))

            scaler = MinMaxScaler()
            X = scaler.fit_transform(X)

            unique, counts = np.unique(y, return_counts=True)

            if np.min(counts) < 2:
                X_train, X_test, y_train, y_test = train_test_split(
                    X, y, test_size=0.3, random_state=42
                )
            else:
                X_train, X_test, y_train, y_test = train_test_split(
                    X, y, test_size=0.3, random_state=42, stratify=y
                )

            d = X.shape[1]

            for algo_name, algo in algos.items():

                sol = algo(lambda s: fitness(X_train, y_train, s), d)
                idx = np.argsort(sol)[-6:]

                Xtr = X_train[:, idx]
                Xte = X_test[:, idx]

                models = get_models(len(idx))

                for model_name, model in models.items():
                    metrics = evaluate(model, Xtr, Xte, y_train, y_test)

                    res = {
                        "Dataset": file,
                        "Algo": algo_name,
                        "Model": model_name,
                        **{k: round(v,4) for k,v in metrics.items()}
                    }

                    results.append(res)
                    print(res)

        except Exception as e:
            print("Error:", file, e)

    df = pd.DataFrame(results)
    df.to_excel("SFP_results_apache_Smote.xlsx", index=False)
    print("\n=== SUMMARY ===")
    #print(df.groupby(["Algo","Model"])["F1"].mean())
    summary = df.groupby(["Algo","Model"])[['Accuracy', 'Precision', 'Recall', 'F1', 'AUC']].mean()
    print(summary)
    summary.to_excel('SFP_summary_apache_Smote.xlsx')


# ==========================
# RUN
# ==========================
run_all()


Processing: ant-1.5.csv
SMOTE applied (k=1)
{'Dataset': 'ant-1.5.csv', 'Algo': 'GA', 'Model': 'XGBoost', 'Accuracy': 0.8523, 'Precision': 0.8628, 'Recall': 0.8523, 'F1': 0.8575, 'AUC': 0.5}
SMOTE applied (k=1)
{'Dataset': 'ant-1.5.csv', 'Algo': 'GA', 'Model': 'LightGBM', 'Accuracy': 0.8295, 'Precision': 0.8495, 'Recall': 0.8295, 'F1': 0.8384, 'AUC': 0.5}
SMOTE applied (k=1)
{'Dataset': 'ant-1.5.csv', 'Algo': 'GA', 'Model': 'CatBoost', 'Accuracy': 0.8295, 'Precision': 0.8564, 'Recall': 0.8295, 'F1': 0.8425, 'AUC': 0.5}
SMOTE applied (k=1)
{'Dataset': 'ant-1.5.csv', 'Algo': 'GA', 'Model': 'HGBoost', 'Accuracy': 0.8523, 'Precision': 0.8628, 'Recall': 0.8523, 'F1': 0.8575, 'AUC': 0.5}
SMOTE applied (k=1)
{'Dataset': 'ant-1.5.csv', 'Algo': 'GA', 'Model': 'MLP', 'Accuracy': 0.4659, 'Precision': 0.9019, 'Recall': 0.4659, 'F1': 0.564, 'AUC': 0.5}
SMOTE applied (k=1)
{'Dataset': 'ant-1.5.csv', 'Algo': 'GA', 'Model': 'ELM', 'Accuracy': 0.8068, 'Precision': 0.8364, 'Recall': 0.8068, 'F1': 0.8197

## Tập Promise + SMOTE

In [7]:
# ============================================================
# Software Fault Prediction - NASA PROMISE (FINAL CLEAN)
# 6 Metaheuristics + 6 Models + Full Metrics + No Warning
# ============================================================

import numpy as np
import pandas as pd
import os
import warnings

# ===== TẮT WARNING =====
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import HistGradientBoostingClassifier

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from imblearn.over_sampling import SMOTE
# ==========================
# ELM
# ==========================
class ELM:
    def __init__(self, input_size, hidden_size=50):
        self.input_size = input_size
        self.hidden_size = hidden_size

    def _act(self, x):
        return 1/(1+np.exp(-x))

    def fit(self, X, y):
        self.W = np.random.randn(self.input_size, self.hidden_size)
        self.b = np.random.randn(self.hidden_size)
        H = self._act(X @ self.W + self.b)
        self.beta = np.linalg.pinv(H) @ y

    def predict(self, X):
        H = self._act(X @ self.W + self.b)
        y = H @ self.beta
        return (y > 0.5).astype(int)

    def predict_proba(self, X):
        H = self._act(X @ self.W + self.b)
        return H @ self.beta


# ==========================
# Load data (FIX binary)
# ==========================
def load_data(path):
    df = pd.read_csv(path).dropna()
    df = df.select_dtypes(include=[np.number])

    X = df.iloc[:, :-1].values
    y = df.iloc[:, -1].values

    # 🔥 Convert defect count → binary
    y = (y > 0).astype(int)

    return X, y


# ==========================
# Fitness
# ==========================

def fitness(X, y, sol):
    # chọn top-6 features
    idx = np.argsort(sol)[-6:]
    X_sel = X[:, idx]

    # ===== scoring (không dùng ML) =====
    scores = np.mean(X_sel, axis=1)

    # normalize về [0,1]
    scores = (scores - scores.min()) / (scores.max() - scores.min() + 1e-9)

    # threshold
    preds = (scores >= 0.5).astype(int)

    # ===== confusion matrix =====
    TP = np.sum((y == 1) & (preds == 1))
    TN = np.sum((y == 0) & (preds == 0))
    FP = np.sum((y == 0) & (preds == 1))
    FN = np.sum((y == 1) & (preds == 0))

    # ===== tránh chia 0 =====
    recall = TP / (TP + FN + 1e-9)
    specificity = TN / (TN + FP + 1e-9)

    return recall + specificity
    

def fitness1(X, y, sol):
    idx = np.argsort(sol)[-6:]
    X_sel = X[:, idx]

    model = MLPClassifier(hidden_layer_sizes=(30,), max_iter=200)
    model.fit(X_sel, y)
    pred = model.predict(X_sel)

    return f1_score(y, pred, average='weighted')


# ==========================
# Metaheuristics
# ==========================
def init_pop(n, d): return np.random.rand(n, d)

def GA(f, d, it=20, n=10):
    P = init_pop(n, d)
    for _ in range(it):
        P = np.array(sorted(P, key=f)[-n//2:])
        C = []
        for _ in range(n//2):
            i, j = np.random.randint(len(P)), np.random.randint(len(P))
            cp = np.random.randint(d)
            C.append(np.concatenate([P[i][:cp], P[j][cp:]]))
        P = np.vstack((P, C))
        P += np.random.normal(0, 0.1, P.shape)
        P = np.clip(P, 0, 1)
    return max(P, key=f)

def PSO(f, d, it=20, n=10):
    X = init_pop(n, d); V = np.zeros_like(X); p = X.copy()
    for _ in range(it):
        for i in range(n):
            if f(X[i]) > f(p[i]): p[i] = X[i]
        g = max(p, key=f)
        for i in range(n):
            r1, r2 = np.random.rand(d), np.random.rand(d)
            V[i] = 0.5*V[i] + 1.5*r1*(p[i]-X[i]) + 1.5*r2*(g-X[i])
            X[i] = np.clip(X[i] + V[i], 0, 1)
    return g

def GWO(f, d, it=20, n=10):
    X = init_pop(n, d)
    for t in range(it):
        X = sorted(X, key=f)
        a = 2 - 2*t/it
        for i in range(n):
            for leader in X[-3:]:
                A = 2*a*np.random.rand(d)-a
                C = 2*np.random.rand(d)
                D = abs(C*leader - X[i])
                X[i] = leader - A*D
        X = np.clip(X, 0, 1)
    return max(X, key=f)

def Squirrel(f, d, it=20, n=10):
    X = init_pop(n, d)
    for _ in range(it):
        best = max(X, key=f)
        for i in range(n):
            X[i] += np.random.rand(d)*(best - X[i])
        X = np.clip(X, 0, 1)
    return best

def MRA(f, d, it=20, n=10):
    X = init_pop(n, d)
    for _ in range(it):
        fit = np.array([f(x) for x in X])
        probs = fit / (fit.sum()+1e-9)
        newX = []
        for _ in range(n):
            i = np.random.choice(range(n), p=probs)
            newX.append(X[i] + np.random.normal(0, 0.1, d))
        X = np.clip(np.array(newX), 0, 1)
    return max(X, key=f)

def SSA(f, d, it=20, n=10):
    X = init_pop(n, d)
    for t in range(it):
        c1 = 2*np.exp(-(4*t/it)**2)
        food = max(X, key=f)
        for i in range(n):
            if i == 0:
                X[i] = food + c1*(np.random.rand(d)-0.5)
            else:
                X[i] = (X[i] + X[i-1]) / 2
        X = np.clip(X, 0, 1)
    return max(X, key=f)

algos = {
    "GA": GA, "Squirrel": Squirrel, "GWO": GWO,
    "MRA": MRA, "PSO": PSO, "SSA": SSA
}


# ==========================
# Models
# ==========================
def get_models(input_dim):
    return {
        "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss'),
        "LightGBM": LGBMClassifier(n_estimators=200, 
                                   learning_rate=0.05,verbose=-1,random_state=42),
        "CatBoost": CatBoostClassifier(iterations=200, verbose=0),
        "HGBoost": HistGradientBoostingClassifier(max_iter=200),
        "MLP": MLPClassifier(hidden_layer_sizes=(50,), max_iter=500, early_stopping=True),
        "ELM": ELM(input_dim)
    }


# ==========================
# SMOTE
# ==========================
def apply_smote(X, y):
    unique, counts = np.unique(y, return_counts=True)

    # nếu dữ liệu quá ít → bỏ qua
    if len(counts) < 2 or np.min(counts) < 2:
        print("SMOTE skipped: too few samples")
        return X, y

    # tự động điều chỉnh k_neighbors
    k = min(5, np.min(counts) - 1)

    try:
        smote = SMOTE(k_neighbors=k, random_state=42)
        X_res, y_res = smote.fit_resample(X, y)
        print(f"SMOTE applied (k={k})")
        return X_res, y_res
    except Exception as e:
        print("SMOTE failed:", e)
        return X, y


# ==========================
# Evaluation
# ==========================
def evaluate(model, X_train, X_test, y_train, y_test):
    
    # ===== SMOTE chỉ áp dụng trên TRAIN =====
    X_train, y_train = apply_smote(X_train, y_train)
    model.fit(X_train, y_train)

    try:
        y_prob = model.predict_proba(X_test)
        if len(y_prob.shape) > 1:
            y_prob = y_prob[:,1]
    except:
        try:
            y_prob = model.decision_function(X_test)
        except:
            y_prob = model.predict(X_test)

    y_pred = model.predict(X_test)

    try:
        auc = roc_auc_score(y_test, y_prob)
    except:
        auc = 0.5

    return {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, average='weighted'),
        "Recall": recall_score(y_test, y_pred, average='weighted'),
        "F1": f1_score(y_test, y_pred, average='weighted'),
        "AUC": auc
    }


# ==========================
# MAIN
# ==========================
def run_all(folder="/kaggle/input/datasets/hovanphi/promisedataset"):
    results = []

    for file in sorted(os.listdir(folder)):
        if not file.endswith(".csv"): continue
        print("\nProcessing:", file)

        try:
            X, y = load_data(os.path.join(folder, file))

            scaler = MinMaxScaler()
            X = scaler.fit_transform(X)

            unique, counts = np.unique(y, return_counts=True)

            if np.min(counts) < 2:
                X_train, X_test, y_train, y_test = train_test_split(
                    X, y, test_size=0.3, random_state=42
                )
            else:
                X_train, X_test, y_train, y_test = train_test_split(
                    X, y, test_size=0.3, random_state=42, stratify=y
                )

            d = X.shape[1]

            for algo_name, algo in algos.items():

                sol = algo(lambda s: fitness(X_train, y_train, s), d)
                idx = np.argsort(sol)[-6:]

                Xtr = X_train[:, idx]
                Xte = X_test[:, idx]

                models = get_models(len(idx))

                for model_name, model in models.items():
                    metrics = evaluate(model, Xtr, Xte, y_train, y_test)

                    res = {
                        "Dataset": file,
                        "Algo": algo_name,
                        "Model": model_name,
                        **{k: round(v,4) for k,v in metrics.items()}
                    }

                    results.append(res)
                    print(res)

        except Exception as e:
            print("Error:", file, e)

    df = pd.DataFrame(results)
    df.to_excel("SFP_results_promis_Smote.xlsx", index=False)
    print("\n=== SUMMARY ===")
    #print(df.groupby(["Algo","Model"])["F1"].mean())
    summary = df.groupby(["Algo","Model"])[['Accuracy', 'Precision', 'Recall', 'F1', 'AUC']].mean()
    print(summary)
    summary.to_excel('SFP_summary_promise_Smote.xlsx')


# ==========================
# RUN
# ==========================
run_all()


Processing: ant-1.3.csv
SMOTE applied (k=5)
{'Dataset': 'ant-1.3.csv', 'Algo': 'GA', 'Model': 'XGBoost', 'Accuracy': 0.7105, 'Precision': 0.7916, 'Recall': 0.7105, 'F1': 0.7408, 'AUC': np.float64(0.6823)}
SMOTE applied (k=5)
{'Dataset': 'ant-1.3.csv', 'Algo': 'GA', 'Model': 'LightGBM', 'Accuracy': 0.7105, 'Precision': 0.7916, 'Recall': 0.7105, 'F1': 0.7408, 'AUC': np.float64(0.6667)}
SMOTE applied (k=5)
{'Dataset': 'ant-1.3.csv', 'Algo': 'GA', 'Model': 'CatBoost', 'Accuracy': 0.6842, 'Precision': 0.7844, 'Recall': 0.6842, 'F1': 0.7205, 'AUC': np.float64(0.6719)}
SMOTE applied (k=5)
{'Dataset': 'ant-1.3.csv', 'Algo': 'GA', 'Model': 'HGBoost', 'Accuracy': 0.6842, 'Precision': 0.7534, 'Recall': 0.6842, 'F1': 0.7132, 'AUC': np.float64(0.7031)}
SMOTE applied (k=5)
{'Dataset': 'ant-1.3.csv', 'Algo': 'GA', 'Model': 'MLP', 'Accuracy': 0.1579, 'Precision': 0.0249, 'Recall': 0.1579, 'F1': 0.0431, 'AUC': np.float64(0.3802)}
SMOTE applied (k=5)
{'Dataset': 'ant-1.3.csv', 'Algo': 'GA', 'Model': 'E

## Tập Nasa + Smote

In [ ]:
# ============================================================
# Software Fault Prediction - NASA PROMISE (FINAL CLEAN)
# 6 Metaheuristics + 6 Models + Full Metrics + No Warning
# ============================================================

import numpy as np
import pandas as pd
import os
import warnings

# ===== TẮT WARNING =====
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import HistGradientBoostingClassifier

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from imblearn.over_sampling import SMOTE
# ==========================
# ELM
# ==========================
class ELM:
    def __init__(self, input_size, hidden_size=50):
        self.input_size = input_size
        self.hidden_size = hidden_size

    def _act(self, x):
        return 1/(1+np.exp(-x))

    def fit(self, X, y):
        self.W = np.random.randn(self.input_size, self.hidden_size)
        self.b = np.random.randn(self.hidden_size)
        H = self._act(X @ self.W + self.b)
        self.beta = np.linalg.pinv(H) @ y

    def predict(self, X):
        H = self._act(X @ self.W + self.b)
        y = H @ self.beta
        return (y > 0.5).astype(int)

    def predict_proba(self, X):
        H = self._act(X @ self.W + self.b)
        return H @ self.beta

# ============================================================
# 🔥 LOAD DATA (FIX ALL NASA ISSUES)
# ============================================================
def load_data(path):
    df = pd.read_csv(path).dropna()

    y = df.iloc[:, -1]

    # label xử lý
    if y.dtype == 'object':
        y = y.astype(str).str.lower().str.strip()
        if set(y.unique()).issubset({'yes', 'no'}):
            y = y.map({'yes': 1, 'no': 0})
        else:
            y = pd.factorize(y)[0]
    else:
        y = (y > 0).astype(int)

    X = df.iloc[:, :-1]
    X = X.select_dtypes(include=[np.number]).values

    return X, y.values


# ==========================
# Fitness
# ==========================
def fitness(X, y, sol):
    # chọn top-6 features
    idx = np.argsort(sol)[-6:]
    X_sel = X[:, idx]

    # ===== scoring (không dùng ML) =====
    scores = np.mean(X_sel, axis=1)

    # normalize về [0,1]
    scores = (scores - scores.min()) / (scores.max() - scores.min() + 1e-9)

    # threshold
    preds = (scores >= 0.5).astype(int)

    # ===== confusion matrix =====
    TP = np.sum((y == 1) & (preds == 1))
    TN = np.sum((y == 0) & (preds == 0))
    FP = np.sum((y == 0) & (preds == 1))
    FN = np.sum((y == 1) & (preds == 0))

    # ===== tránh chia 0 =====
    recall = TP / (TP + FN + 1e-9)
    specificity = TN / (TN + FP + 1e-9)

    return recall + specificity


def fitness1(X, y, sol):
    idx = np.argsort(sol)[-6:]
    X_sel = X[:, idx]

    model = MLPClassifier(hidden_layer_sizes=(30,), max_iter=200)
    model.fit(X_sel, y)
    pred = model.predict(X_sel)

    return f1_score(y, pred, average='weighted')


# ==========================
# Metaheuristics
# ==========================
def init_pop(n, d): return np.random.rand(n, d)

def GA(f, d, it=20, n=10):
    P = init_pop(n, d)
    for _ in range(it):
        P = np.array(sorted(P, key=f)[-n//2:])
        C = []
        for _ in range(n//2):
            i, j = np.random.randint(len(P)), np.random.randint(len(P))
            cp = np.random.randint(d)
            C.append(np.concatenate([P[i][:cp], P[j][cp:]]))
        P = np.vstack((P, C))
        P += np.random.normal(0, 0.1, P.shape)
        P = np.clip(P, 0, 1)
    return max(P, key=f)

def PSO(f, d, it=20, n=10):
    X = init_pop(n, d); V = np.zeros_like(X); p = X.copy()
    for _ in range(it):
        for i in range(n):
            if f(X[i]) > f(p[i]): p[i] = X[i]
        g = max(p, key=f)
        for i in range(n):
            r1, r2 = np.random.rand(d), np.random.rand(d)
            V[i] = 0.5*V[i] + 1.5*r1*(p[i]-X[i]) + 1.5*r2*(g-X[i])
            X[i] = np.clip(X[i] + V[i], 0, 1)
    return g

def GWO(f, d, it=20, n=10):
    X = init_pop(n, d)
    for t in range(it):
        X = sorted(X, key=f)
        a = 2 - 2*t/it
        for i in range(n):
            for leader in X[-3:]:
                A = 2*a*np.random.rand(d)-a
                C = 2*np.random.rand(d)
                D = abs(C*leader - X[i])
                X[i] = leader - A*D
        X = np.clip(X, 0, 1)
    return max(X, key=f)

def Squirrel(f, d, it=20, n=10):
    X = init_pop(n, d)
    for _ in range(it):
        best = max(X, key=f)
        for i in range(n):
            X[i] += np.random.rand(d)*(best - X[i])
        X = np.clip(X, 0, 1)
    return best

def MRA(f, d, it=20, n=10):
    X = init_pop(n, d)
    for _ in range(it):
        fit = np.array([f(x) for x in X])
        probs = fit / (fit.sum()+1e-9)
        newX = []
        for _ in range(n):
            i = np.random.choice(range(n), p=probs)
            newX.append(X[i] + np.random.normal(0, 0.1, d))
        X = np.clip(np.array(newX), 0, 1)
    return max(X, key=f)

def SSA(f, d, it=20, n=10):
    X = init_pop(n, d)
    for t in range(it):
        c1 = 2*np.exp(-(4*t/it)**2)
        food = max(X, key=f)
        for i in range(n):
            if i == 0:
                X[i] = food + c1*(np.random.rand(d)-0.5)
            else:
                X[i] = (X[i] + X[i-1]) / 2
        X = np.clip(X, 0, 1)
    return max(X, key=f)

algos = {
    "GA": GA, "Squirrel": Squirrel, "GWO": GWO,
    "MRA": MRA, "PSO": PSO, "SSA": SSA
}


# ==========================
# Models
# ==========================
def get_models(input_dim):
    return {
        "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss'),
        "LightGBM": LGBMClassifier(n_estimators=200, 
                                   learning_rate=0.05,verbose=-1,random_state=42),
        "CatBoost": CatBoostClassifier(iterations=200, verbose=0),
        "HGBoost": HistGradientBoostingClassifier(max_iter=200),
        "MLP": MLPClassifier(hidden_layer_sizes=(50,), max_iter=500, early_stopping=True),
        "ELM": ELM(input_dim)
    }


# ==========================
# SMOTE
# ==========================
def apply_smote(X, y):
    unique, counts = np.unique(y, return_counts=True)

    # nếu dữ liệu quá ít → bỏ qua
    if len(counts) < 2 or np.min(counts) < 2:
        print("SMOTE skipped: too few samples")
        return X, y

    # tự động điều chỉnh k_neighbors
    k = min(5, np.min(counts) - 1)

    try:
        smote = SMOTE(k_neighbors=k, random_state=42)
        X_res, y_res = smote.fit_resample(X, y)
        print(f"SMOTE applied (k={k})")
        return X_res, y_res
    except Exception as e:
        print("SMOTE failed:", e)
        return X, y


# ==========================
# Evaluation
# ==========================
def evaluate(model, X_train, X_test, y_train, y_test):
    
    # ===== SMOTE chỉ áp dụng trên TRAIN =====
    X_train, y_train = apply_smote(X_train, y_train)
    model.fit(X_train, y_train)

    try:
        y_prob = model.predict_proba(X_test)
        if len(y_prob.shape) > 1:
            y_prob = y_prob[:,1]
    except:
        try:
            y_prob = model.decision_function(X_test)
        except:
            y_prob = model.predict(X_test)

    y_pred = model.predict(X_test)

    try:
        auc = roc_auc_score(y_test, y_prob)
    except:
        auc = 0.5

    return {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, average='weighted'),
        "Recall": recall_score(y_test, y_pred, average='weighted'),
        "F1": f1_score(y_test, y_pred, average='weighted'),
        "AUC": auc
    }


# ==========================
# MAIN
# ==========================
def run_all(folder="/kaggle/input/datasets/hovanphi/nasadataset"):
    results = []

    for file in sorted(os.listdir(folder)):
        if not file.endswith(".csv"): continue
        print("\nProcessing:", file)

        try:
            X, y = load_data(os.path.join(folder, file))

            scaler = MinMaxScaler()
            X = scaler.fit_transform(X)

            unique, counts = np.unique(y, return_counts=True)

            if np.min(counts) < 2:
                X_train, X_test, y_train, y_test = train_test_split(
                    X, y, test_size=0.3, random_state=42
                )
            else:
                X_train, X_test, y_train, y_test = train_test_split(
                    X, y, test_size=0.3, random_state=42, stratify=y
                )

            d = X.shape[1]

            for algo_name, algo in algos.items():

                sol = algo(lambda s: fitness(X_train, y_train, s), d)
                idx = np.argsort(sol)[-6:]

                Xtr = X_train[:, idx]
                Xte = X_test[:, idx]

                models = get_models(len(idx))

                for model_name, model in models.items():
                    metrics = evaluate(model, Xtr, Xte, y_train, y_test)

                    res = {
                        "Dataset": file,
                        "Algo": algo_name,
                        "Model": model_name,
                        **{k: round(v,4) for k,v in metrics.items()}
                    }

                    results.append(res)
                    print(res)

        except Exception as e:
            print("Error:", file, e)

    df = pd.DataFrame(results)
    df.to_excel("SFP_results_nasa_Smote.xlsx", index=False)
    print("\n=== SUMMARY ===")
    #print(df.groupby(["Algo","Model"])["F1"].mean())
    summary = df.groupby(["Algo","Model"])[['Accuracy', 'Precision', 'Recall', 'F1', 'AUC']].mean()
    print(summary)
    summary.to_excel('SFP_summary_nasa_Smote.xlsx')


# ==========================
# RUN
# ==========================
run_all()


Processing: cm1.csv
SMOTE applied (k=5)
{'Dataset': 'cm1.csv', 'Algo': 'GA', 'Model': 'XGBoost', 'Accuracy': 0.7867, 'Precision': 0.8368, 'Recall': 0.7867, 'F1': 0.8092, 'AUC': np.float64(0.6701)}
SMOTE applied (k=5)
{'Dataset': 'cm1.csv', 'Algo': 'GA', 'Model': 'LightGBM', 'Accuracy': 0.78, 'Precision': 0.8525, 'Recall': 0.78, 'F1': 0.8102, 'AUC': np.float64(0.6696)}
SMOTE applied (k=5)
{'Dataset': 'cm1.csv', 'Algo': 'GA', 'Model': 'CatBoost', 'Accuracy': 0.7333, 'Precision': 0.8275, 'Recall': 0.7333, 'F1': 0.7738, 'AUC': np.float64(0.64)}
SMOTE applied (k=5)
{'Dataset': 'cm1.csv', 'Algo': 'GA', 'Model': 'HGBoost', 'Accuracy': 0.7733, 'Precision': 0.8429, 'Recall': 0.7733, 'F1': 0.8032, 'AUC': np.float64(0.6301)}
SMOTE applied (k=5)
{'Dataset': 'cm1.csv', 'Algo': 'GA', 'Model': 'MLP', 'Accuracy': 0.6, 'Precision': 0.8522, 'Recall': 0.6, 'F1': 0.6798, 'AUC': np.float64(0.5709)}
SMOTE applied (k=5)
{'Dataset': 'cm1.csv', 'Algo': 'GA', 'Model': 'ELM', 'Accuracy': 0.6867, 'Precision': 0.

# Giữ nguyên features
## Tập Promise

In [27]:
# ============================================================
# Software Fault Prediction - NASA PROMISE (FINAL CLEAN)
# 6 Metaheuristics + 6 Models + Full Metrics + No Warning
# ============================================================

import numpy as np
import pandas as pd
import os
import warnings

# ===== TẮT WARNING =====
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import HistGradientBoostingClassifier

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# ==========================
# ELM
# ==========================
class ELM:
    def __init__(self, input_size, hidden_size=50):
        self.input_size = input_size
        self.hidden_size = hidden_size

    def _act(self, x):
        return 1/(1+np.exp(-x))

    def fit(self, X, y):
        self.W = np.random.randn(self.input_size, self.hidden_size)
        self.b = np.random.randn(self.hidden_size)
        H = self._act(X @ self.W + self.b)
        self.beta = np.linalg.pinv(H) @ y

    def predict(self, X):
        H = self._act(X @ self.W + self.b)
        y = H @ self.beta
        return (y > 0.5).astype(int)

    def predict_proba(self, X):
        H = self._act(X @ self.W + self.b)
        return H @ self.beta

# ============================================================
# 🔥 LOAD DATA (FIX ALL NASA ISSUES)
# ============================================================
def load_data(path):
    df = pd.read_csv(path).dropna()

    y = df.iloc[:, -1]

    # label xử lý
    if y.dtype == 'object':
        y = y.astype(str).str.lower().str.strip()
        if set(y.unique()).issubset({'yes', 'no'}):
            y = y.map({'yes': 1, 'no': 0})
        else:
            y = pd.factorize(y)[0]
    else:
        y = (y > 0).astype(int)

    X = df.iloc[:, :-1]
    X = X.select_dtypes(include=[np.number]).values

    return X, y.values


# ==========================
# Fitness
# ==========================
def fitness(X, y, sol):
    idx = np.argsort(sol)[-6:]
    X_sel = X[:, idx]

    model = MLPClassifier(hidden_layer_sizes=(30,), max_iter=200)
    model.fit(X_sel, y)
    pred = model.predict(X_sel)

    return f1_score(y, pred, average='weighted')


# ==========================
# Models
# ==========================
def get_models(input_dim):
    return {
        "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss'),
        "LightGBM": LGBMClassifier(n_estimators=200, 
                                   learning_rate=0.05,verbose=-1,random_state=42),
        "CatBoost": CatBoostClassifier(iterations=200, verbose=0),
        "HGBoost": HistGradientBoostingClassifier(max_iter=200),
        "MLP": MLPClassifier(hidden_layer_sizes=(50,), max_iter=500, early_stopping=True),
        "ELM": ELM(input_dim)
    }



# ==========================
# Evaluation
# ==========================
def evaluate(model, X_train, X_test, y_train, y_test):
    
    model.fit(X_train, y_train)

    try:
        y_prob = model.predict_proba(X_test)
        if len(y_prob.shape) > 1:
            y_prob = y_prob[:,1]
    except:
        try:
            y_prob = model.decision_function(X_test)
        except:
            y_prob = model.predict(X_test)

    y_pred = model.predict(X_test)

    try:
        auc = roc_auc_score(y_test, y_prob)
    except:
        auc = 0.5

    return {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, average='weighted'),
        "Recall": recall_score(y_test, y_pred, average='weighted'),
        "F1": f1_score(y_test, y_pred, average='weighted'),
        "AUC": auc
    }


# ==========================
# MAIN
# ==========================
def run_all(folder="/kaggle/input/datasets/hovanphi/promisedataset"):
    results = []

    for file in sorted(os.listdir(folder)):
        if not file.endswith(".csv"): continue
        print("\nProcessing:", file)

        try:
            X, y = load_data(os.path.join(folder, file))

            scaler = MinMaxScaler()
            X = scaler.fit_transform(X)

            unique, counts = np.unique(y, return_counts=True)

            if np.min(counts) < 2:
                X_train, X_test, y_train, y_test = train_test_split(
                    X, y, test_size=0.3, random_state=42
                )
            else:
                X_train, X_test, y_train, y_test = train_test_split(
                    X, y, test_size=0.3, random_state=42, stratify=y
                )

            Xtr = X_train
            Xte = X_test

            models = get_models(X_train.shape[1])

            for model_name, model in models.items():
                metrics = evaluate(model, Xtr, Xte, y_train, y_test)

                res = {
                    "Dataset": file,
                    "Model": model_name,
                    **{k: round(v,4) for k,v in metrics.items()}
                }

                results.append(res)
                print(res)

        except Exception as e:
            print("Error:", file, e)

    df = pd.DataFrame(results)
    df.to_excel("SFP_results_promise_AllFeatures.xlsx", index=False)
    print("\n=== SUMMARY ===")
    #print(df.groupby(["Algo","Model"])["F1"].mean())
    summary = df.groupby(["Dataset","Model"])[['Accuracy', 'Precision', 'Recall', 'F1', 'AUC']].mean()
    print(summary)
    summary.to_excel('SFP_summary_promise_AllFeatures.xlsx')


# ==========================
# RUN
# ==========================
run_all()


Processing: ant-1.3.csv
{'Dataset': 'ant-1.3.csv', 'Model': 'XGBoost', 'Accuracy': 0.8421, 'Precision': 0.8421, 'Recall': 0.8421, 'F1': 0.8421, 'AUC': np.float64(0.7552)}
{'Dataset': 'ant-1.3.csv', 'Model': 'LightGBM', 'Accuracy': 0.7632, 'Precision': 0.7786, 'Recall': 0.7632, 'F1': 0.7704, 'AUC': np.float64(0.7344)}
{'Dataset': 'ant-1.3.csv', 'Model': 'CatBoost', 'Accuracy': 0.8947, 'Precision': 0.9064, 'Recall': 0.8947, 'F1': 0.8715, 'AUC': np.float64(0.7292)}
{'Dataset': 'ant-1.3.csv', 'Model': 'HGBoost', 'Accuracy': 0.7632, 'Precision': 0.7786, 'Recall': 0.7632, 'F1': 0.7704, 'AUC': np.float64(0.7031)}
{'Dataset': 'ant-1.3.csv', 'Model': 'MLP', 'Accuracy': 0.8421, 'Precision': 0.7091, 'Recall': 0.8421, 'F1': 0.7699, 'AUC': np.float64(0.4688)}
{'Dataset': 'ant-1.3.csv', 'Model': 'ELM', 'Accuracy': 0.7105, 'Precision': 0.7916, 'Recall': 0.7105, 'F1': 0.7408, 'AUC': np.float64(0.7656)}

Processing: ant-1.4.csv
{'Dataset': 'ant-1.4.csv', 'Model': 'XGBoost', 'Accuracy': 0.8148, 'Precis

## Nasa

In [26]:
# ============================================================
# Software Fault Prediction - NASA PROMISE (FINAL CLEAN)
# 6 Metaheuristics + 6 Models + Full Metrics + No Warning
# ============================================================

import numpy as np
import pandas as pd
import os
import warnings

# ===== TẮT WARNING =====
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import HistGradientBoostingClassifier

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# ==========================
# ELM
# ==========================
class ELM:
    def __init__(self, input_size, hidden_size=50):
        self.input_size = input_size
        self.hidden_size = hidden_size

    def _act(self, x):
        return 1/(1+np.exp(-x))

    def fit(self, X, y):
        self.W = np.random.randn(self.input_size, self.hidden_size)
        self.b = np.random.randn(self.hidden_size)
        H = self._act(X @ self.W + self.b)
        self.beta = np.linalg.pinv(H) @ y

    def predict(self, X):
        H = self._act(X @ self.W + self.b)
        y = H @ self.beta
        return (y > 0.5).astype(int)

    def predict_proba(self, X):
        H = self._act(X @ self.W + self.b)
        return H @ self.beta

# ============================================================
# 🔥 LOAD DATA (FIX ALL NASA ISSUES)
# ============================================================
def load_data(path):
    df = pd.read_csv(path).dropna()

    y = df.iloc[:, -1]

    # label xử lý
    if y.dtype == 'object':
        y = y.astype(str).str.lower().str.strip()
        if set(y.unique()).issubset({'yes', 'no'}):
            y = y.map({'yes': 1, 'no': 0})
        else:
            y = pd.factorize(y)[0]
    else:
        y = (y > 0).astype(int)

    X = df.iloc[:, :-1]
    X = X.select_dtypes(include=[np.number]).values

    return X, y.values


# ==========================
# Fitness
# ==========================
def fitness(X, y, sol):
    idx = np.argsort(sol)[-6:]
    X_sel = X[:, idx]

    model = MLPClassifier(hidden_layer_sizes=(30,), max_iter=200)
    model.fit(X_sel, y)
    pred = model.predict(X_sel)

    return f1_score(y, pred, average='weighted')


# ==========================
# Models
# ==========================
def get_models(input_dim):
    return {
        "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss'),
        "LightGBM": LGBMClassifier(n_estimators=200, 
                                   learning_rate=0.05,verbose=-1,random_state=42),
        "CatBoost": CatBoostClassifier(iterations=200, verbose=0),
        "HGBoost": HistGradientBoostingClassifier(max_iter=200),
        "MLP": MLPClassifier(hidden_layer_sizes=(50,), max_iter=500, early_stopping=True),
        "ELM": ELM(input_dim)
    }



# ==========================
# Evaluation
# ==========================
def evaluate(model, X_train, X_test, y_train, y_test):
    
    model.fit(X_train, y_train)

    try:
        y_prob = model.predict_proba(X_test)
        if len(y_prob.shape) > 1:
            y_prob = y_prob[:,1]
    except:
        try:
            y_prob = model.decision_function(X_test)
        except:
            y_prob = model.predict(X_test)

    y_pred = model.predict(X_test)

    try:
        auc = roc_auc_score(y_test, y_prob)
    except:
        auc = 0.5

    return {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, average='weighted'),
        "Recall": recall_score(y_test, y_pred, average='weighted'),
        "F1": f1_score(y_test, y_pred, average='weighted'),
        "AUC": auc
    }


# ==========================
# MAIN
# ==========================
def run_all(folder="/kaggle/input/datasets/hovanphi/nasadataset"):
    results = []

    for file in sorted(os.listdir(folder)):
        if not file.endswith(".csv"): continue
        print("\nProcessing:", file)

        try:
            X, y = load_data(os.path.join(folder, file))

            scaler = MinMaxScaler()
            X = scaler.fit_transform(X)

            unique, counts = np.unique(y, return_counts=True)

            if np.min(counts) < 2:
                X_train, X_test, y_train, y_test = train_test_split(
                    X, y, test_size=0.3, random_state=42
                )
            else:
                X_train, X_test, y_train, y_test = train_test_split(
                    X, y, test_size=0.3, random_state=42, stratify=y
                )

            Xtr = X_train
            Xte = X_test

            models = get_models(X_train.shape[1])

            for model_name, model in models.items():
                metrics = evaluate(model, Xtr, Xte, y_train, y_test)

                res = {
                    "Dataset": file,
                    "Model": model_name,
                    **{k: round(v,4) for k,v in metrics.items()}
                }

                results.append(res)
                print(res)

        except Exception as e:
            print("Error:", file, e)

    df = pd.DataFrame(results)
    df.to_excel("SFP_results_Nasa_AllFeatures.xlsx", index=False)
    print("\n=== SUMMARY ===")
    #print(df.groupby(["Algo","Model"])["F1"].mean())
    summary = df.groupby(["Dataset","Model"])[['Accuracy', 'Precision', 'Recall', 'F1', 'AUC']].mean()
    print(summary)
    summary.to_excel('SFP_summary_Nasa_AllFeatures.xlsx')


# ==========================
# RUN
# ==========================
run_all()


Processing: cm1.csv
{'Dataset': 'cm1.csv', 'Model': 'XGBoost', 'Accuracy': 0.8667, 'Precision': 0.8496, 'Recall': 0.8667, 'F1': 0.8574, 'AUC': np.float64(0.6025)}
{'Dataset': 'cm1.csv', 'Model': 'LightGBM', 'Accuracy': 0.8733, 'Precision': 0.8616, 'Recall': 0.8733, 'F1': 0.867, 'AUC': np.float64(0.6188)}
{'Dataset': 'cm1.csv', 'Model': 'CatBoost', 'Accuracy': 0.8867, 'Precision': 0.8088, 'Recall': 0.8867, 'F1': 0.8459, 'AUC': np.float64(0.6321)}
{'Dataset': 'cm1.csv', 'Model': 'HGBoost', 'Accuracy': 0.8467, 'Precision': 0.8319, 'Recall': 0.8467, 'F1': 0.839, 'AUC': np.float64(0.6356)}
{'Dataset': 'cm1.csv', 'Model': 'MLP', 'Accuracy': 0.8733, 'Precision': 0.8075, 'Recall': 0.8733, 'F1': 0.8391, 'AUC': np.float64(0.4089)}
{'Dataset': 'cm1.csv', 'Model': 'ELM', 'Accuracy': 0.86, 'Precision': 0.8063, 'Recall': 0.86, 'F1': 0.8323, 'AUC': np.float64(0.683)}

Processing: jm1.csv
{'Dataset': 'jm1.csv', 'Model': 'XGBoost', 'Accuracy': 0.847, 'Precision': 0.8169, 'Recall': 0.847, 'F1': 0.8189,

## Nasa + SMAOTE

In [ ]:
# ============================================================
# Software Fault Prediction - NASA PROMISE (FINAL CLEAN)
# 6 Metaheuristics + 6 Models + Full Metrics + No Warning
# ============================================================

import numpy as np
import pandas as pd
import os
import warnings

# ===== TẮT WARNING =====
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import HistGradientBoostingClassifier

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from imblearn.over_sampling import SMOTE
# ==========================
# ELM
# ==========================
class ELM:
    def __init__(self, input_size, hidden_size=50):
        self.input_size = input_size
        self.hidden_size = hidden_size

    def _act(self, x):
        return 1/(1+np.exp(-x))

    def fit(self, X, y):
        self.W = np.random.randn(self.input_size, self.hidden_size)
        self.b = np.random.randn(self.hidden_size)
        H = self._act(X @ self.W + self.b)
        self.beta = np.linalg.pinv(H) @ y

    def predict(self, X):
        H = self._act(X @ self.W + self.b)
        y = H @ self.beta
        return (y > 0.5).astype(int)

    def predict_proba(self, X):
        H = self._act(X @ self.W + self.b)
        return H @ self.beta

# ============================================================
# 🔥 LOAD DATA (FIX ALL NASA ISSUES)
# ============================================================
def load_data(path):
    df = pd.read_csv(path).dropna()

    y = df.iloc[:, -1]

    # label xử lý
    if y.dtype == 'object':
        y = y.astype(str).str.lower().str.strip()
        if set(y.unique()).issubset({'yes', 'no'}):
            y = y.map({'yes': 1, 'no': 0})
        else:
            y = pd.factorize(y)[0]
    else:
        y = (y > 0).astype(int)

    X = df.iloc[:, :-1]
    X = X.select_dtypes(include=[np.number]).values

    return X, y.values


# ==========================
# Fitness
# ==========================
def fitness(X, y, sol):
    idx = np.argsort(sol)[-6:]
    X_sel = X[:, idx]

    model = MLPClassifier(hidden_layer_sizes=(30,), max_iter=200)
    model.fit(X_sel, y)
    pred = model.predict(X_sel)

    return f1_score(y, pred, average='weighted')


# ==========================
# Models
# ==========================
def get_models(input_dim):
    return {
        "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss'),
        "LightGBM": LGBMClassifier(n_estimators=200, 
                                   learning_rate=0.05,verbose=-1,random_state=42),
        "CatBoost": CatBoostClassifier(iterations=200, verbose=0),
        "HGBoost": HistGradientBoostingClassifier(max_iter=200),
        "MLP": MLPClassifier(hidden_layer_sizes=(50,), max_iter=500, early_stopping=True),
        "ELM": ELM(input_dim)
    }


# ==========================
# SMOTE
# ==========================
def apply_smote(X, y):
    unique, counts = np.unique(y, return_counts=True)

    # nếu dữ liệu quá ít → bỏ qua
    if len(counts) < 2 or np.min(counts) < 2:
        print("SMOTE skipped: too few samples")
        return X, y

    # tự động điều chỉnh k_neighbors
    k = min(5, np.min(counts) - 1)

    try:
        smote = SMOTE(k_neighbors=k, random_state=42)
        X_res, y_res = smote.fit_resample(X, y)
        print(f"SMOTE applied (k={k})")
        return X_res, y_res
    except Exception as e:
        print("SMOTE failed:", e)
        return X, y


# ==========================
# Evaluation
# ==========================
def evaluate(model, X_train, X_test, y_train, y_test):
    
    # ===== SMOTE chỉ áp dụng trên TRAIN =====
    X_train, y_train = apply_smote(X_train, y_train)
    model.fit(X_train, y_train)

    try:
        y_prob = model.predict_proba(X_test)
        if len(y_prob.shape) > 1:
            y_prob = y_prob[:,1]
    except:
        try:
            y_prob = model.decision_function(X_test)
        except:
            y_prob = model.predict(X_test)

    y_pred = model.predict(X_test)

    try:
        auc = roc_auc_score(y_test, y_prob)
    except:
        auc = 0.5

    return {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, average='weighted'),
        "Recall": recall_score(y_test, y_pred, average='weighted'),
        "F1": f1_score(y_test, y_pred, average='weighted'),
        "AUC": auc
    }


# ==========================
# MAIN
# ==========================
def run_all(folder="/kaggle/input/datasets/hovanphi/promise"):
    results = []

    for file in sorted(os.listdir(folder)):
        if not file.endswith(".csv"): continue
        print("\nProcessing:", file)

        try:
            X, y = load_data(os.path.join(folder, file))

            scaler = MinMaxScaler()
            X = scaler.fit_transform(X)

            unique, counts = np.unique(y, return_counts=True)

            if np.min(counts) < 2:
                X_train, X_test, y_train, y_test = train_test_split(
                    X, y, test_size=0.3, random_state=42
                )
            else:
                X_train, X_test, y_train, y_test = train_test_split(
                    X, y, test_size=0.3, random_state=42, stratify=y
                )

            Xtr = X_train
            Xte = X_test

            models = get_models(X_train.shape[1])

            for model_name, model in models.items():
                metrics = evaluate(model, Xtr, Xte, y_train, y_test)

                res = {
                    "Dataset": file,
                    "Model": model_name,
                    **{k: round(v,4) for k,v in metrics.items()}
                }

                results.append(res)
                print(res)

        except Exception as e:
            print("Error:", file, e)

    df = pd.DataFrame(results)
    df.to_excel("SFP_results_promise_AllFeatures.xlsx", index=False)
    print("\n=== SUMMARY ===")
    #print(df.groupby(["Algo","Model"])["F1"].mean())
    summary = df.groupby(["Dataset","Model"])[['Accuracy', 'Precision', 'Recall', 'F1', 'AUC']].mean()
    print(summary)
    summary.to_excel('SFP_summary_promise_AllFeatures.xlsx')


# ==========================
# RUN
# ==========================
run_all()